# Amharic ACOS Extraction — Stage 1 (Aspect + Opinion Term Extraction)

**Before running:** in Kaggle, open *Settings* (right panel) and:
1. Turn **Internet** ON (needed to download the pretrained encoder from Hugging Face).
2. Turn **Accelerator** to **GPU** (T4 x2 or P100).
3. Add your TSV files as a Kaggle Dataset (Add Data → Upload) containing
   `amharic_quad_train.tsv`, `amharic_quad_test.tsv`, `amharic_quad_dev.tsv`,
   then adjust `RAW_DIR` below to match the input path Kaggle assigns it
   (usually `/kaggle/input/<your-dataset-name>/`).

In [ ]:
!pip install -q transformers>=4.40 tqdm

In [ ]:
RAW_DIR = "/kaggle/input/amharic-acos"  # <-- change to your uploaded dataset's input path
TRAIN_TSV = f"{RAW_DIR}/amharic_quad_train.tsv"
TEST_TSV  = f"{RAW_DIR}/amharic_quad_test.tsv"
DEV_TSV   = f"{RAW_DIR}/amharic_quad_dev.tsv"

MODEL_NAME = "Davlan/afro-xlmr-base"   # swap to "rasyosef/bert-small-amharic" for the lightweight baseline
OUTPUT_DIR = "/kaggle/working/stage1_ckpt"

## 1. Word-level BIO tagging utilities (`bio_labels.py`)

In [ ]:
"""
Word-level BIO tag construction and decoding for aspect/opinion term extraction.
Pure Python, no tokenizer dependency -- this is tested locally. The subword
alignment step (word-level BIO -> subword-level BIO using a HF fast tokenizer's
word_ids()) lives in align.py and needs to run where transformers is installed.
"""
from typing import List, Tuple


def build_word_bio(n_tokens: int, spans: List[Tuple[int, int]]) -> List[str]:
    """spans: list of (start, end) word-index spans, end EXCLUSIVE, -1 spans skipped
    (implicit terms have no span to tag). Overlapping spans from different quads
    are simply unioned onto the same tag sequence -- pairing is resolved in stage 2."""
    tags = ["O"] * n_tokens
    for start, end in spans:
        if start == -1 or end == -1:
            continue
        if not (0 <= start < end <= n_tokens):
            raise ValueError(f"Span ({start},{end}) out of range for {n_tokens} tokens")
        tags[start] = "B"
        for i in range(start + 1, end):
            tags[i] = "I"
    return tags


def decode_bio_spans(tags: List[str]) -> List[Tuple[int, int]]:
    """Inverse of build_word_bio: BIO tag sequence -> list of (start, end) spans."""
    spans = []
    start = None
    for i, tag in enumerate(tags + ["O"]):  # sentinel to close trailing span
        if tag == "B":
            if start is not None:
                spans.append((start, i))
            start = i
        elif tag == "O":
            if start is not None:
                spans.append((start, i))
                start = None
        # tag == "I": continue current span (if start is None, treat as noise -> ignore)
    return spans


# (local self-test omitted in notebook version)

In [ ]:
%%writefile bio_labels.py
"""
Word-level BIO tag construction and decoding for aspect/opinion term extraction.
Pure Python, no tokenizer dependency -- this is tested locally. The subword
alignment step (word-level BIO -> subword-level BIO using a HF fast tokenizer's
word_ids()) lives in align.py and needs to run where transformers is installed.
"""
from typing import List, Tuple


def build_word_bio(n_tokens: int, spans: List[Tuple[int, int]]) -> List[str]:
    """spans: list of (start, end) word-index spans, end EXCLUSIVE, -1 spans skipped
    (implicit terms have no span to tag). Overlapping spans from different quads
    are simply unioned onto the same tag sequence -- pairing is resolved in stage 2."""
    tags = ["O"] * n_tokens
    for start, end in spans:
        if start == -1 or end == -1:
            continue
        if not (0 <= start < end <= n_tokens):
            raise ValueError(f"Span ({start},{end}) out of range for {n_tokens} tokens")
        tags[start] = "B"
        for i in range(start + 1, end):
            tags[i] = "I"
    return tags


def decode_bio_spans(tags: List[str]) -> List[Tuple[int, int]]:
    """Inverse of build_word_bio: BIO tag sequence -> list of (start, end) spans."""
    spans = []
    start = None
    for i, tag in enumerate(tags + ["O"]):  # sentinel to close trailing span
        if tag == "B":
            if start is not None:
                spans.append((start, i))
            start = i
        elif tag == "O":
            if start is not None:
                spans.append((start, i))
                start = None
        # tag == "I": continue current span (if start is None, treat as noise -> ignore)
    return spans

## 2. Subword alignment (`align.py`)

In [ ]:
%%writefile align.py
"""
Word-level BIO tags -> subword-level BIO tags, using a HF fast tokenizer's word_ids().
Requires: transformers (fast tokenizer). Run in your GPU environment, not this sandbox.
"""
from typing import List, Dict

LABEL2ID = {"O": 0, "B": 1, "I": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
IGNORE_INDEX = -100  # HF loss ignores this automatically


def align_labels_to_subwords(word_ids: List[int], word_tags: List[str]) -> List[int]:
    """
    word_ids: output of tokenizer(...).word_ids(batch_index=i) -- one entry per
        subword token, giving the source word index (or None for special tokens).
    word_tags: BIO tags at word level (from bio_labels.build_word_bio).

    Rule: first subword of a word gets the word's tag; continuation subwords of
    a 'B' word get 'I' (so a multi-subword aspect term reads B,I,I,... not B,B,B);
    continuation subwords of an 'I' word stay 'I'; special tokens get IGNORE_INDEX.
    """
    label_ids = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            label_ids.append(IGNORE_INDEX)
        elif wid != prev_word_id:
            label_ids.append(LABEL2ID[word_tags[wid]])
        else:
            # continuation subword of the same word
            tag = word_tags[wid]
            label_ids.append(LABEL2ID["I"] if tag in ("B", "I") else LABEL2ID["O"])
        prev_word_id = wid
    return label_ids


def decode_subword_predictions(word_ids: List[int], pred_ids: List[int]) -> List[str]:
    """Inverse: take the model's per-subword predictions and reduce back to one
    tag per word (using the first subword's prediction for each word -- the
    standard convention for token classification with subword tokenizers)."""
    word_tags: Dict[int, str] = {}
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None:
            continue
        if wid not in word_tags:
            word_tags[wid] = ID2LABEL[pid]
    n_words = max(word_tags) + 1 if word_tags else 0
    return [word_tags.get(i, "O") for i in range(n_words)]

## 3. Data prep — parse TSVs, clean category schema, write JSONL

In [ ]:
%%writefile data_prep.py
"""
Data preparation for Amharic ACOS (Aspect-Category-Opinion-Sentiment) extraction.

Input format (TSV), one sentence per line:
    <text>\t<quad1>\t<quad2>...
where each quad is a single field, space-separated:
    "aStart,aEnd CATEGORY sentiment oStart,oEnd"
Spans are whitespace-token indices into `text.split()`. "-1,-1" = implicit (no span).
Sentiment: 0=neutral, 1=positive, 2=negative.

This script:
  1. Parses raw TSV into structured records.
  2. Canonicalizes the category schema (fixes cross-domain label overlap).
  3. Writes clean JSONL files (train/dev/test) ready for tokenizer alignment.
  4. Prints before/after category distribution so the mapping is auditable.
"""
import json
import argparse
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import List, Tuple

SENTIMENT_MAP = {"0": "NEUTRAL", "1": "POSITIVE", "2": "NEGATIVE"}

# Manual merges for near-duplicate fine-grained tags that don't cleanly collapse
# by dropping the domain prefix alone.
MANUAL_CATEGORY_MERGE = {
    "CRIME_SERVICES": "CRIME",
    "INFRASTRUCTURE": "UTILITIES",  # from PUBLIC_SERVICES#INFRASTRUCTURE (1 example)
}

MIN_CATEGORY_COUNT = 30  # categories with fewer than this many train instances -> OTHER


@dataclass
class Quad:
    a_start: int
    a_end: int
    category: str
    sentiment: str
    o_start: int
    o_end: int

    @property
    def aspect_implicit(self) -> bool:
        return self.a_start == -1

    @property
    def opinion_implicit(self) -> bool:
        return self.o_start == -1


@dataclass
class Example:
    text: str
    tokens: List[str]
    quads: List[Quad]


def parse_quad_field(field: str) -> Quad:
    a_span, cat, sent, o_span = field.strip().split(" ")
    a_start, a_end = (int(x) for x in a_span.split(","))
    o_start, o_end = (int(x) for x in o_span.split(","))
    return Quad(a_start, a_end, cat, SENTIMENT_MAP[sent], o_start, o_end)


def parse_tsv(path: str) -> List[Example]:
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            parts = line.split("\t")
            text = parts[0]
            tokens = text.split()
            quads = [parse_quad_field(q) for q in parts[1:] if q.strip()]
            examples.append(Example(text=text, tokens=tokens, quads=quads))
    return examples


def canonical_category(raw_cat: str) -> str:
    """Drop the domain prefix (e.g. GOVERNANCE#TRANSPARENCY -> TRANSPARENCY),
    which merges tags that were duplicated across domains due to schema overlap."""
    fine = raw_cat.split("#", 1)[1] if "#" in raw_cat else raw_cat
    return MANUAL_CATEGORY_MERGE.get(fine, fine)


def build_category_mapping(train_examples: List[Example]) -> dict:
    """Build raw_category -> final_category mapping, applying the MIN_CATEGORY_COUNT
    threshold (computed on TRAIN only, then reused for dev/test for consistency)."""
    counts = Counter()
    raw_to_canonical = {}
    for ex in train_examples:
        for q in ex.quads:
            canon = canonical_category(q.category)
            raw_to_canonical[q.category] = canon
            counts[canon] += 1

    final_mapping = {}
    for raw_cat, canon in raw_to_canonical.items():
        final_mapping[raw_cat] = canon if counts[canon] >= MIN_CATEGORY_COUNT else "OTHER"
    return final_mapping


def apply_mapping(examples: List[Example], mapping: dict) -> List[Example]:
    for ex in examples:
        for q in ex.quads:
            q.category = mapping.get(q.category, "OTHER")
    return examples


def report_distribution(examples: List[Example], label: str):
    counts = Counter(q.category for ex in examples for q in ex.quads)
    total = sum(counts.values())
    print(f"\n--- {label}: category distribution after canonicalization ({total} quads) ---")
    for cat, c in counts.most_common():
        print(f"  {cat}: {c} ({100*c/total:.1f}%)")


def write_jsonl(examples: List[Example], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for ex in examples:
            rec = {
                "text": ex.text,
                "tokens": ex.tokens,
                "quads": [asdict(q) for q in ex.quads],
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", required=True)
    ap.add_argument("--test", required=True)
    ap.add_argument("--dev", default=None)
    ap.add_argument("--out_dir", default="./prepared")
    args = ap.parse_args()

    import os
    os.makedirs(args.out_dir, exist_ok=True)

    train_examples = parse_tsv(args.train)
    test_examples = parse_tsv(args.test)
    dev_examples = parse_tsv(args.dev) if args.dev else None

    mapping = build_category_mapping(train_examples)

    print("=== Category mapping (raw -> final) ===")
    for raw, final in sorted(mapping.items()):
        flag = "  <-- merged/renamed" if raw.split("#", 1)[-1] != final else ""
        print(f"  {raw}  ->  {final}{flag}")

    train_examples = apply_mapping(train_examples, mapping)
    test_examples = apply_mapping(test_examples, mapping)
    if dev_examples:
        dev_examples = apply_mapping(dev_examples, mapping)

    report_distribution(train_examples, "TRAIN")
    report_distribution(test_examples, "TEST")

    write_jsonl(train_examples, os.path.join(args.out_dir, "train.jsonl"))
    write_jsonl(test_examples, os.path.join(args.out_dir, "test.jsonl"))
    if dev_examples:
        write_jsonl(dev_examples, os.path.join(args.out_dir, "dev.jsonl"))

    final_categories = sorted(set(mapping.values()))
    with open(os.path.join(args.out_dir, "label_space.json"), "w", encoding="utf-8") as f:
        json.dump({
            "categories": final_categories,
            "sentiments": ["NEUTRAL", "POSITIVE", "NEGATIVE"],
            "category_mapping": mapping,
        }, f, ensure_ascii=False, indent=2)

    print(f"\nWrote prepared data to {args.out_dir}/  ({len(final_categories)} final categories)")


if __name__ == "__main__":
    main()

In [ ]:
!python data_prep.py \
    --train $TRAIN_TSV \
    --test $TEST_TSV \
    --dev $DEV_TSV \
    --out_dir /kaggle/working/prepared

*(If the `!python ... $VAR` substitution above doesn't pick up your Python variables, just hardcode the three paths directly in the command.)*

## 4. Stage 1 dataset (`dataset_tagging.py`)

In [ ]:
%%writefile dataset_tagging.py
"""
Stage 1 dataset: joint Aspect Term Extraction (ATE) + Opinion Term Extraction (OTE)
via BIO tagging, sharing one encoder with two independent linear heads.
Run where transformers/torch are installed.
"""
import json
import torch
from torch.utils.data import Dataset

from bio_labels import build_word_bio
from align import align_labels_to_subwords, IGNORE_INDEX


class TaggingDataset(Dataset):
    def __init__(self, jsonl_path: str, tokenizer, max_length: int = 256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.records = []
        with open(jsonl_path, encoding="utf-8") as f:
            for line in f:
                self.records.append(json.loads(line))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        tokens = rec["tokens"]
        n = len(tokens)

        a_spans = [(q["a_start"], q["a_end"]) for q in rec["quads"]]
        o_spans = [(q["o_start"], q["o_end"]) for q in rec["quads"]]
        a_word_tags = build_word_bio(n, a_spans)
        o_word_tags = build_word_bio(n, o_spans)

        enc = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
        )
        word_ids = enc.word_ids(batch_index=0)

        a_label_ids = align_labels_to_subwords(word_ids, a_word_tags)
        o_label_ids = align_labels_to_subwords(word_ids, o_word_tags)

        item = {k: torch.tensor(v) for k, v in enc.items() if k != "overflow_to_sample_mapping"}
        item["aspect_labels"] = torch.tensor(a_label_ids)
        item["opinion_labels"] = torch.tensor(o_label_ids)
        return item


def collate_fn(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}

## 5. Stage 1 model (`model_tagging.py`)

In [ ]:
%%writefile model_tagging.py
"""
Shared-encoder, dual-head token classification model for joint ATE + OTE.
"""
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

NUM_BIO_LABELS = 3  # O, B, I


class JointTaggingModel(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.aspect_head = nn.Linear(hidden, NUM_BIO_LABELS)
        self.opinion_head = nn.Linear(hidden, NUM_BIO_LABELS)

    def forward(self, input_ids, attention_mask, aspect_labels=None, opinion_labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        seq_out = self.dropout(out.last_hidden_state)  # (B, T, H)

        aspect_logits = self.aspect_head(seq_out)   # (B, T, 3)
        opinion_logits = self.opinion_head(seq_out)  # (B, T, 3)

        loss = None
        if aspect_labels is not None and opinion_labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss_a = loss_fct(aspect_logits.view(-1, NUM_BIO_LABELS), aspect_labels.view(-1))
            loss_o = loss_fct(opinion_logits.view(-1, NUM_BIO_LABELS), opinion_labels.view(-1))
            loss = loss_a + loss_o

        return {
            "loss": loss,
            "aspect_logits": aspect_logits,
            "opinion_logits": opinion_logits,
        }

## 6. Training script (`train_tagging.py`)

In [ ]:
%%writefile train_tagging.py
"""
Stage 1 training: joint Aspect Term Extraction + Opinion Term Extraction.

Usage (in your GPU environment):
    pip install -r requirements.txt
    python train_tagging.py \
        --train ../prepared/train.jsonl \
        --test ../prepared/test.jsonl \
        --model_name Davlan/afro-xlmr-base \
        --output_dir ./stage1_tagging_ckpt \
        --epochs 8 --batch_size 16 --lr 3e-5

Swap --model_name to rasyosef/bert-small-amharic for the lightweight
Amharic-native baseline comparison.
"""
import argparse
import json
import os
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

from dataset_tagging import TaggingDataset, collate_fn
from model_tagging import JointTaggingModel
from bio_labels import decode_bio_spans
from align import decode_subword_predictions


def span_prf(pred_spans_per_ex, gold_spans_per_ex):
    tp = fp = fn = 0
    for preds, golds in zip(pred_spans_per_ex, gold_spans_per_ex):
        preds_set, golds_set = set(preds), set(golds)
        tp += len(preds_set & golds_set)
        fp += len(preds_set - golds_set)
        fn += len(golds_set - preds_set)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


@torch.no_grad()
def evaluate(model, dataset, tokenizer, device, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collate_fn)

    all_a_pred, all_a_gold, all_o_pred, all_o_gold = [], [], [], []
    rec_idx = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        out = model(input_ids=input_ids, attention_mask=attn)
        a_pred_ids = out["aspect_logits"].argmax(-1).cpu().tolist()
        o_pred_ids = out["opinion_logits"].argmax(-1).cpu().tolist()

        bsz = input_ids.size(0)
        for i in range(bsz):
            rec = dataset.records[rec_idx]
            rec_idx += 1
            enc = tokenizer(rec["tokens"], is_split_into_words=True,
                             truncation=True, max_length=dataset.max_length)
            word_ids = enc.word_ids(batch_index=0)

            a_word_tags = decode_subword_predictions(word_ids, a_pred_ids[i][:len(word_ids)])
            o_word_tags = decode_subword_predictions(word_ids, o_pred_ids[i][:len(word_ids)])

            all_a_pred.append(decode_bio_spans(a_word_tags))
            all_o_pred.append(decode_bio_spans(o_word_tags))
            all_a_gold.append([(q["a_start"], q["a_end"]) for q in rec["quads"] if q["a_start"] != -1])
            all_o_gold.append([(q["o_start"], q["o_end"]) for q in rec["quads"] if q["o_start"] != -1])

    return {
        "aspect": span_prf(all_a_pred, all_a_gold),
        "opinion": span_prf(all_o_pred, all_o_gold),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--train", required=True)
    ap.add_argument("--test", required=True)
    ap.add_argument("--model_name", default="Davlan/afro-xlmr-base")
    ap.add_argument("--output_dir", default="./stage1_tagging_ckpt")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--max_length", type=int, default=256)
    ap.add_argument("--warmup_ratio", type=float, default=0.06)
    args = ap.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(args.output_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    train_ds = TaggingDataset(args.train, tokenizer, max_length=args.max_length)
    test_ds = TaggingDataset(args.test, tokenizer, max_length=args.max_length)

    model = JointTaggingModel(args.model_name).to(device)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, collate_fn=collate_fn)
    total_steps = len(train_loader) * args.epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * args.warmup_ratio), num_training_steps=total_steps
    )

    best_f1 = -1.0
    for epoch in range(args.epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{args.epochs}")
        for batch in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix(loss=loss.item())

        metrics = evaluate(model, test_ds, tokenizer, device)
        print(f"\n[epoch {epoch+1}] aspect F1={metrics['aspect']['f1']:.4f} "
              f"(P={metrics['aspect']['precision']:.4f} R={metrics['aspect']['recall']:.4f})  "
              f"opinion F1={metrics['opinion']['f1']:.4f} "
              f"(P={metrics['opinion']['precision']:.4f} R={metrics['opinion']['recall']:.4f})")

        avg_f1 = (metrics["aspect"]["f1"] + metrics["opinion"]["f1"]) / 2
        if avg_f1 > best_f1:
            best_f1 = avg_f1
            torch.save(model.state_dict(), os.path.join(args.output_dir, "best_model.pt"))
            tokenizer.save_pretrained(args.output_dir)
            with open(os.path.join(args.output_dir, "best_metrics.json"), "w") as f:
                json.dump(metrics, f, indent=2)
            print(f"  -> new best (avg F1={avg_f1:.4f}), checkpoint saved")

    print(f"\nDone. Best avg span F1 = {best_f1:.4f}. Checkpoint: {args.output_dir}/best_model.pt")


if __name__ == "__main__":
    main()

## 7. Run training

In [ ]:
!python train_tagging.py \
    --train /kaggle/working/prepared/train.jsonl \
    --test /kaggle/working/prepared/test.jsonl \
    --model_name "$MODEL_NAME" \
    --output_dir "$OUTPUT_DIR" \
    --epochs 8 --batch_size 16 --lr 3e-5

## What's next
This trains **Stage 1 only**: joint Aspect Term Extraction + Opinion Term Extraction
(explicit spans only — implicit aspects/opinions, ~32%/19% of your data, are not
covered here and need a separate stage). Next stages: aspect–opinion pairing,
category classification, sentiment classification, implicit detection, and
end-to-end quad-level evaluation.